# Notebook Overview — Extract Frequency-Domain Features

## Purpose

This notebook extracts **frequency-domain features** from preprocessed images for a selected dataset subset (**train** or **test**).

It computes FFT-based spectral energy, radial profile, and log-spectrum features, and outputs structured feature vectors for downstream classification.

---

## Inputs

Required inputs:

* Metadata CSV file (train or test):
  * `metadata/splits/<train_metadata_filename>`
  * `metadata/splits/<test_metadata_filename>`

* Preprocessed image archive:
  * `releases/preprocessed/All_Sources_preprocessed.zip`

Shared configuration:

* `src/project_config.py`

---

## Execution Model

* One subset (**train** or **test**) is processed per run
* Metadata is loaded and validated for consistency
* Image file paths are constructed and verified
* Preprocessed images are extracted to the local runtime if needed
* Frequency-domain features are computed using centered 2D FFT analysis
* Feature extraction is performed sequentially across all images
* Errors during processing are handled and reported
* Output feature vectors are validated and saved to CSV
* Processing is deterministic and reproducible

---

## Outputs

**Frequency Feature CSV (Train)**  
`metadata/features/<train_frequency_features_filename>`

**Frequency Feature CSV (Test)**  
`metadata/features/<test_frequency_features_filename>`

Each output file contains:

* Metadata columns:
  * `filename`
  * `class_label`
  * `source_dataset`
  * `subset`

* Frequency-domain feature columns:
  * Low Frequency Energy Ratio  
  * High Frequency Energy Ratio  
  * Radial Mean  
  * Radial Std  
  * Radial Entropy  
  * Spectral Centroid  
  * Spectral Bandwidth  
  * Log Spectrum Std  

**Expected Behavior**

* One feature vector is generated per input image  
* Output row count matches input metadata unless errors occur  
* Feature values are computed consistently across all images  
* Output is ready for downstream feature vector construction and normalization  

---
---

### 🔷 Step 1 — Startup: Environment and Verification

- Mount Google Drive for access to the preprocessed image ZIP archive
- Clone the GitHub repository if needed
- Add the project `src` directory to the Python path
- Import shared configuration values from `project_config.py`
- Define metadata, image extraction, and frequency feature output paths
- Select the train or test subset for feature extraction
- Verify that required metadata and ZIP input files are present
- Extract preprocessed images into the local runtime if needed
- Verify that extracted PNG images are available
- Optionally display key configuration values when `VERBOSE=True`

---

In [ ]:
# ============================================
# Step 1: Startup — Environment and Verification
# ============================================

# -------------------------------------------------
# Import required libraries
# -------------------------------------------------
import os
import sys
import zipfile
from pathlib import Path

# -------------------------------------------------
# Notebook display control
# -------------------------------------------------
VERBOSE = True   # Set to False to reduce detailed output

# -------------------------------------------------
# Mount Google Drive for access to release ZIP archive
# -------------------------------------------------
from google.colab import drive
drive.mount("/content/drive")

# -------------------------------------------------
# Clone GitHub repository if needed
# -------------------------------------------------
REPO_URL = "https://github.com/pgailinas/dip-ai-image-detection.git"
REPO_DIR = Path("/content/dip-ai-image-detection")

if not REPO_DIR.exists():
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

# -------------------------------------------------
# Add src directory to Python path
# -------------------------------------------------
SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# -------------------------------------------------
# Import shared project configuration
# -------------------------------------------------
from project_config import (
    TRAIN_SUBSET,
    TEST_SUBSET,
    TRAIN_METADATA_PATH,
    TEST_METADATA_PATH,
    TRAIN_FREQUENCY_FEATURES_PATH,
    TEST_FREQUENCY_FEATURES_PATH,
    PREPROCESSED_ZIP_PATHS,
    PROCESSED_DATA_DIR,
    FEATURES_METADATA_DIR,
)

# -------------------------------------------------
# Convert configured paths to Path objects
# -------------------------------------------------
TRAIN_METADATA_FILE = Path(TRAIN_METADATA_PATH)
TEST_METADATA_FILE = Path(TEST_METADATA_PATH)

TRAIN_OUTPUT_FILE = Path(TRAIN_FREQUENCY_FEATURES_PATH)
TEST_OUTPUT_FILE = Path(TEST_FREQUENCY_FEATURES_PATH)

PREPROCESSED_ZIP = Path(PREPROCESSED_ZIP_PATHS["all_sources"])

EXTRACTED_IMAGE_DIR = Path(PROCESSED_DATA_DIR) / "images"
FEATURES_DIR = Path(FEATURES_METADATA_DIR)

FEATURES_DIR.mkdir(parents=True, exist_ok=True)
EXTRACTED_IMAGE_DIR.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# Select subset to process
# -------------------------------------------------
SUBSET_NAME = TRAIN_SUBSET   # change to TEST_SUBSET for test run

if SUBSET_NAME == TRAIN_SUBSET:
    METADATA_FILE = TRAIN_METADATA_FILE
    OUTPUT_FILE = TRAIN_OUTPUT_FILE
elif SUBSET_NAME == TEST_SUBSET:
    METADATA_FILE = TEST_METADATA_FILE
    OUTPUT_FILE = TEST_OUTPUT_FILE
else:
    raise ValueError(f"Invalid SUBSET_NAME: {SUBSET_NAME}")

# -------------------------------------------------
# Verify required input files
# -------------------------------------------------
print("Verifying required inputs...\n")

required_files = [
    METADATA_FILE,
    PREPROCESSED_ZIP,
]

for file_path in required_files:
    if not file_path.exists():
        raise FileNotFoundError(f"Missing required file: {file_path}")

print("Required input files are present.")

# -------------------------------------------------
# Optional verbose display of configuration values
# -------------------------------------------------
if VERBOSE:
    print(f"Subset selected:  {SUBSET_NAME}")
    print(f"Metadata file:    {METADATA_FILE}")
    print(f"Preprocessed ZIP: {PREPROCESSED_ZIP}")
    print(f"Extracted images: {EXTRACTED_IMAGE_DIR}")
    print(f"Output file:      {OUTPUT_FILE}")

# -------------------------------------------------
# Extract preprocessed image ZIP if needed
# -------------------------------------------------
existing_png_count = len(list(EXTRACTED_IMAGE_DIR.glob("*.png")))

if existing_png_count > 0:
    print(f"\nExtracted image directory already contains {existing_png_count} PNG files.")
    print("Skipping extraction.")
else:
    print("\nExtracting All_Sources_preprocessed.zip to local runtime...")
    with zipfile.ZipFile(PREPROCESSED_ZIP, "r") as zip_ref:
        zip_ref.extractall(EXTRACTED_IMAGE_DIR)
    print("Extraction complete.")

# -------------------------------------------------
# Verify extracted image files
# -------------------------------------------------
image_files = list(EXTRACTED_IMAGE_DIR.glob("*.png"))

if len(image_files) == 0:
    raise FileNotFoundError(
        f"No PNG files found in extracted image directory: {EXTRACTED_IMAGE_DIR}"
    )

print(f"Found {len(image_files)} extracted PNG files.")
print("\nStartup complete.")



### 🔷 Step 2 — Load Metadata

- Load the selected train or test metadata CSV file
- Verify that required metadata columns are present
- Confirm that the metadata subset matches the selected subset
- Build local image paths from metadata filenames
- Verify that all metadata-referenced image files exist
- Display class distribution for the selected subset
- Optionally display source distribution and sample rows when `VERBOSE=True`

---


In [ ]:
# ============================================
# Step 2: Load Metadata
# ============================================

# -------------------------------------------------
# Import required library
# -------------------------------------------------
import pandas as pd

# -------------------------------------------------
# Load selected subset metadata
# -------------------------------------------------
df = pd.read_csv(METADATA_FILE)

print(f"Loaded metadata for subset: {SUBSET_NAME}")
print(f"Metadata shape: {df.shape}")

# -------------------------------------------------
# Verify required metadata columns
# -------------------------------------------------
required_columns = ["filename", "class_label", "source_dataset", "subset"]
missing_columns = [col for col in required_columns if col not in df.columns]

if missing_columns:
    raise ValueError(f"Metadata file is missing required columns: {missing_columns}")

print("Required metadata columns verified.")

# -------------------------------------------------
# Verify subset consistency
# -------------------------------------------------
unique_subsets = sorted(df["subset"].dropna().unique().tolist())

if unique_subsets != [SUBSET_NAME]:
    raise ValueError(
        f"Metadata subset mismatch. Expected only '{SUBSET_NAME}', found: {unique_subsets}"
    )

print(f"Subset column verified: {unique_subsets}")

# -------------------------------------------------
# Build image paths from filenames
# -------------------------------------------------
df["image_path"] = df["filename"].apply(lambda x: EXTRACTED_IMAGE_DIR / x)

# -------------------------------------------------
# Verify metadata-referenced image files
# -------------------------------------------------
missing_images = df.loc[~df["image_path"].apply(lambda p: p.exists()), "filename"].tolist()

if missing_images:
    raise FileNotFoundError(
        "Some image files referenced by metadata were not found in the extracted image directory.\n"
        f"First missing files: {missing_images[:10]}"
    )

print("All metadata-referenced image files were found.")

# -------------------------------------------------
# Display class distribution
# -------------------------------------------------
print("\nClass distribution:")
print(df["class_label"].value_counts())

# -------------------------------------------------
# Optional verbose display of metadata summary
# -------------------------------------------------
if VERBOSE:
    print("\nSource distribution:")
    print(df["source_dataset"].value_counts())

    print("\nSample rows:")
    display(df.head())



### 🔷 Step 3 — Frequency Feature Helper Functions

- Define centered 2D FFT computation
- Define magnitude, log-magnitude, and power spectrum computation
- Define radial mean power profile computation
- Define entropy computation for the radial power profile
- Define low-, mid-, and high-frequency energy ratio computation
- Define spectral centroid and spectral bandwidth computations
- Define frequency-domain feature extraction function
- Compute spectral energy, radial profile, and log-spectrum features

---

In [ ]:
# ============================================
# Step 3: Frequency Feature Helper Functions
# ============================================

# -------------------------------------------------
# Import required libraries
# -------------------------------------------------
import numpy as np
from scipy.stats import entropy

# -------------------------------------------------
# Compute centered 2D FFT
# -------------------------------------------------
def compute_fft(img):
    # Compute 2D FFT and shift zero-frequency component to center
    F = np.fft.fft2(img)
    F_shift = np.fft.fftshift(F)
    return F_shift

# -------------------------------------------------
# Magnitude, log-magnitude, and power spectrum
# -------------------------------------------------
def compute_magnitude_spectrum(F_shift):
    # Compute frequency magnitude representations
    mag = np.abs(F_shift)
    log_mag = np.log1p(mag)
    power = mag ** 2
    return mag, log_mag, power

# -------------------------------------------------
# Radial mean power profile
# -------------------------------------------------
def radial_profile(power_spectrum):
    # Average power values by radial distance from spectrum center
    h, w = power_spectrum.shape
    cy, cx = h // 2, w // 2

    y, x = np.indices((h, w))
    r = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)
    r = r.astype(np.int32)

    max_r = r.max()
    radial_mean = np.zeros(max_r + 1, dtype=np.float32)

    for i in range(max_r + 1):
        mask = (r == i)
        if np.any(mask):
            radial_mean[i] = power_spectrum[mask].mean()

    return radial_mean

# -------------------------------------------------
# Entropy of radial profile
# -------------------------------------------------
def radial_entropy(radial_vals):
    # Compute entropy of radial power distribution
    hist, _ = np.histogram(radial_vals, bins=64)
    hist = hist.astype(np.float64)
    hist = hist / (hist.sum() + 1e-12)
    hist = np.clip(hist, 1e-12, None)
    return float(entropy(hist, base=2))

# -------------------------------------------------
# Frequency band energy ratios
# -------------------------------------------------
def frequency_band_ratios(power_spectrum):
    # Compute low-, mid-, and high-frequency energy ratios
    h, w = power_spectrum.shape
    cy, cx = h // 2, w // 2

    y, x = np.indices((h, w))
    r = np.sqrt((x - cx) ** 2 + (y - cy) ** 2)

    max_r = np.max(r)

    low_mask = r <= max_r * 0.2
    mid_mask = (r > max_r * 0.2) & (r <= max_r * 0.5)
    high_mask = r > max_r * 0.5

    total_energy = power_spectrum.sum() + 1e-12

    low_ratio = float(power_spectrum[low_mask].sum() / total_energy)
    mid_ratio = float(power_spectrum[mid_mask].sum() / total_energy)
    high_ratio = float(power_spectrum[high_mask].sum() / total_energy)

    return low_ratio, mid_ratio, high_ratio

# -------------------------------------------------
# Spectral centroid
# -------------------------------------------------
def spectral_centroid(radial_vals):
    # Compute weighted mean frequency radius
    indices = np.arange(len(radial_vals))
    total = radial_vals.sum() + 1e-12
    centroid = float((indices * radial_vals).sum() / total)
    return centroid

# -------------------------------------------------
# Spectral bandwidth
# -------------------------------------------------
def spectral_bandwidth(radial_vals, centroid):
    # Compute spread of radial energy around spectral centroid
    indices = np.arange(len(radial_vals))
    total = radial_vals.sum() + 1e-12
    variance = ((indices - centroid) ** 2 * radial_vals).sum() / total
    return float(np.sqrt(variance))

# -------------------------------------------------
# Extract frequency-domain features
# -------------------------------------------------
def extract_frequency_features(img):
    # Compute FFT-based spectral representations
    F_shift = compute_fft(img)
    mag, log_mag, power = compute_magnitude_spectrum(F_shift)

    # Compute radial power profile
    radial_vals = radial_profile(power)

    # Compute frequency band energy ratios
    low_ratio, mid_ratio, high_ratio = frequency_band_ratios(power)

    # Radial profile statistics
    r_mean = float(np.mean(radial_vals))
    r_std = float(np.std(radial_vals))
    r_entropy = radial_entropy(radial_vals)

    # Spectral centroid and bandwidth
    centroid = spectral_centroid(radial_vals)
    bandwidth = spectral_bandwidth(radial_vals, centroid)

    # Log-spectrum variability
    log_std = float(np.std(log_mag))

    # Assemble feature vector
    features = {
        "Low Frequency Energy Ratio": low_ratio,
        "High Frequency Energy Ratio": high_ratio,
        "Radial Mean": r_mean,
        "Radial Std": r_std,
        "Radial Entropy": r_entropy,
        "Spectral Centroid": centroid,
        "Spectral Bandwidth": bandwidth,
        "Log Spectrum Std": log_std,
    }

    return features, log_mag, power, radial_vals

print("Frequency helper functions defined.")



### 🔷 Step 4 — Preview and Validate Sample Image

- Select a sample image from the loaded metadata
- Verify that the image file exists and can be loaded
- Confirm expected image format (grayscale, 256×256)
- Compute frequency-domain features for the sample image
- Optionally display sample metadata and feature values when `VERBOSE=True`
- Visualize the image, log magnitude spectrum, log power spectrum, and radial profile

---

In [ ]:
# ============================================
# Step 4: Preview and Validate Sample Image
# ============================================

# -------------------------------------------------
# Import required libraries
# -------------------------------------------------
import cv2
import matplotlib.pyplot as plt

# -------------------------------------------------
# Select a sample image from metadata
# -------------------------------------------------
sample_row = df.iloc[0]
sample_path = sample_row["image_path"]

# -------------------------------------------------
# Optional verbose display of sample metadata
# -------------------------------------------------
if VERBOSE:
    print("Sample metadata row:")
    for key, value in sample_row.items():
        print(f"{key}: {value}")

    print(f"\nSample image path: {sample_path}")

if not sample_path.exists():
    raise FileNotFoundError(f"Sample image not found: {sample_path}")

# -------------------------------------------------
# Load sample image in grayscale
# -------------------------------------------------
sample_img = cv2.imread(str(sample_path), cv2.IMREAD_GRAYSCALE)

if sample_img is None:
    raise ValueError(f"Could not load sample image: {sample_path}")

# -------------------------------------------------
# Verify image properties
# -------------------------------------------------
if VERBOSE:
    print(f"Loaded sample image shape: {sample_img.shape}")
    print(f"Loaded sample image dtype: {sample_img.dtype}")

if len(sample_img.shape) != 2:
    raise ValueError("Expected grayscale image with 2 dimensions.")

if sample_img.shape != (256, 256):
    raise ValueError(f"Expected image shape (256, 256), found {sample_img.shape}")

print("Sample image format verified.")

# -------------------------------------------------
# Compute frequency features for sample image
# -------------------------------------------------
sample_features, log_mag, power, radial_vals = extract_frequency_features(sample_img)

# -------------------------------------------------
# Optional verbose display of sample features
# -------------------------------------------------
if VERBOSE:
    print("\nSample frequency-domain features:")
    for key, value in sample_features.items():
        print(f"{key}: {value:.6f}")

    print(f"\nNumber of features extracted: {len(sample_features)}")

# -------------------------------------------------
# Display sample image and frequency visualizations
# -------------------------------------------------
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(sample_img, cmap="gray")
axes[0].set_title("Sample Image")
axes[0].axis("off")

axes[1].imshow(log_mag, cmap="gray")
axes[1].set_title("Log Magnitude Spectrum")
axes[1].axis("off")

axes[2].imshow(np.log1p(power), cmap="gray")
axes[2].set_title("Log Power Spectrum")
axes[2].axis("off")

axes[3].plot(radial_vals)
axes[3].set_title("Radial Profile")
axes[3].set_xlabel("Radius")
axes[3].set_ylabel("Mean Power")

plt.tight_layout()
plt.show()



### 🔷 Step 5 — Extract Frequency-Domain Features for Subset

- Iterate over all images in the selected subset
- Load each image in grayscale format
- Compute frequency-domain feature vectors for each image
- Assemble metadata and feature values into structured records
- Handle and track errors for failed image processing
- Convert extracted records into a DataFrame
- Verify expected output columns and ordering
- Report extraction summary and processing statistics
- Optionally display sample output rows when `VERBOSE=True`

---

In [ ]:
# ============================================
# Step 5: Extract Frequency-Domain Features for Subset
# ============================================

# -------------------------------------------------
# Import progress tracking utility
# -------------------------------------------------
from tqdm.notebook import tqdm

# -------------------------------------------------
# Extract features for all images in selected subset
# -------------------------------------------------
records = []
error_count = 0

print(f"Beginning frequency-domain feature extraction for subset: {SUBSET_NAME}")
print(f"Total images to process: {len(df)}\n")

for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {SUBSET_NAME} images"):
    image_path = row["image_path"]

    try:
        # Load image in grayscale
        img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)

        if img is None:
            raise ValueError(f"Could not load image: {image_path}")

        # Extract frequency-domain features
        features, _, _, _ = extract_frequency_features(img)

        # Build output record
        record = {
            "filename": row["filename"],
            "class_label": row["class_label"],
            "source_dataset": row["source_dataset"],
            "subset": row["subset"],
        }
        record.update(features)
        records.append(record)

    except Exception as e:
        error_count += 1
        if VERBOSE:
            print(f"Skipping {row['filename']} due to error: {e}")

# -------------------------------------------------
# Convert records to DataFrame
# -------------------------------------------------
frequency_features_df = pd.DataFrame(records)

# -------------------------------------------------
# Verify output structure and column order
# -------------------------------------------------
expected_columns = [
    "filename",
    "class_label",
    "source_dataset",
    "subset",
    "Low Frequency Energy Ratio",
    "High Frequency Energy Ratio",
    "Radial Mean",
    "Radial Std",
    "Radial Entropy",
    "Spectral Centroid",
    "Spectral Bandwidth",
    "Log Spectrum Std",
]

missing_columns = [col for col in expected_columns if col not in frequency_features_df.columns]
if missing_columns:
    raise ValueError(f"Missing expected output columns: {missing_columns}")

frequency_features_df = frequency_features_df[expected_columns]

# -------------------------------------------------
# Summary of extraction results
# -------------------------------------------------
print("\nFrequency-domain feature extraction complete.")
print(f"Output shape: {frequency_features_df.shape}")
print(f"Processed images: {len(frequency_features_df)}")
print(f"Expected images:  {len(df)}")

if error_count > 0:
    print(f"Skipped images:   {error_count}")

if len(frequency_features_df) != len(df):
    print("\nWARNING: Some images were skipped during processing.")

# -------------------------------------------------
# Optional verbose display of output sample
# -------------------------------------------------
if VERBOSE:
    print("\nSample output rows:")
    display(frequency_features_df.head())



### 🔷 Step 6 — Save Output

- Create output directory if it does not exist
- Save frequency feature DataFrame to CSV file
- Confirm output file location
- Verify that the output file was created successfully
- Validate that saved row count matches extracted data

---

In [ ]:
# ============================================
# Step 6: Save Output
# ============================================

# -------------------------------------------------
# Ensure output directory exists
# -------------------------------------------------
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# Save frequency feature DataFrame to CSV
# -------------------------------------------------
frequency_features_df.to_csv(OUTPUT_FILE, index=False)

# -------------------------------------------------
# Confirm save location
# -------------------------------------------------
print(f"Saved frequency features to: {OUTPUT_FILE}")

# -------------------------------------------------
# Verify output file was created
# -------------------------------------------------
if not OUTPUT_FILE.exists():
    raise FileNotFoundError(f"Output file was not created: {OUTPUT_FILE}")

# -------------------------------------------------
# Verify saved row count
# -------------------------------------------------
saved_df = pd.read_csv(OUTPUT_FILE)

if len(saved_df) != len(frequency_features_df):
    raise ValueError(
        f"Saved row count mismatch. Expected {len(frequency_features_df)}, found {len(saved_df)}"
    )

print(f"Verified saved file row count: {len(saved_df)}")



### 🔷 Step 7 — Final Summary and Next Step

- Display summary of frequency-domain feature extraction results
- Report input size, output size, and saved file location
- List extracted frequency-domain feature columns
- Provide guidance for next execution step (train → test → feature vector construction)
- Display formatted completion message

---

In [ ]:
# ============================================
# Step 7: Final Summary and Next Step
# ============================================

# -------------------------------------------------
# Import display utilities
# -------------------------------------------------
from IPython.display import display, HTML

# -------------------------------------------------
# Print execution summary
# -------------------------------------------------
print("Frequency-domain feature extraction completed successfully.\n")

print(f"Subset processed : {SUBSET_NAME}")
print(f"Input rows       : {len(df)}")
print(f"Output rows      : {len(frequency_features_df)}")
print(f"Output columns   : {len(frequency_features_df.columns)}")
print(f"Saved file       : {OUTPUT_FILE}")

# -------------------------------------------------
# Display feature column names
# -------------------------------------------------
print("\nFeature columns:")
for col in frequency_features_df.columns[4:]:
    print(f"  - {col}")

# -------------------------------------------------
# Display next-step guidance message
# -------------------------------------------------
if SUBSET_NAME == TRAIN_SUBSET:
    message = """
    <b>Next Step:</b><br>
    Set <code>SUBSET_NAME = TEST_SUBSET</code> in Step 1 and rerun this notebook to generate <b>test</b> frequency-domain features.
    """
    border_color = "#ff9800"
    bg_color = "#fff3e0"
else:
    message = f"""
    <b>Current Run Complete:</b><br>
    This run generated <b>{OUTPUT_FILE.name}</b> for the <b>test</b> subset.<br><br>
    After both train and test frequency feature files are available, proceed to
    <b>05_Build_Feature_Vectors.ipynb</b>.
    """
    border_color = "#4CAF50"
    bg_color = "#E8F5E9"

# -------------------------------------------------
# Render formatted summary message
# -------------------------------------------------
display(HTML(f"""
<div style="
    padding: 15px;
    border: 2px solid {border_color};
    background-color: {bg_color};
    border-radius: 8px;
    font-size: 16px;
">
{message}
</div>
"""))

